# Options Pricing: Black-Scholes vs Monte Carlo

**Quantra | classical-quant-engine | Month 1**

This notebook benchmarks the closed-form Black-Scholes model against a Monte Carlo simulation engine for European options pricing. We examine:

1. ATM/ITM/OTM price comparison across strikes
2. Full Greeks surface for calls
3. Implied volatility surface (vol × maturity grid)
4. Convergence of MC price as a function of simulation count

All parameters are calibrated to approximate Nifty 50 index option characteristics.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import cm

from src.options.black_scholes import call_price, put_price, greeks
from src.options.monte_carlo import price_both

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})

## 1. Base Parameters

Calibrated to approximate Nifty 50 index options (NIFTY50 spot ~22,000, 3-month expiry).

In [ ]:
S     = 22_000.0   # Nifty 50 spot (approximate)
T     = 0.25       # 3-month expiry
r     = 0.065      # RBI repo rate proxy
sigma = 0.18       # ~18% annualised vol (Nifty 50 historical)
N_SIM = 200_000

print(f"Spot: {S:,.0f} | T: {T} yr | r: {r*100:.1f}% | σ: {sigma*100:.0f}%")

## 2. Price vs Strike — BS vs Monte Carlo Comparison Table

In [ ]:
strikes = np.arange(20_000, 24_500, 500)
rows = []

for K in strikes:
    bs_call = call_price(S, K, T, r, sigma)
    bs_put  = put_price(S, K, T, r, sigma)
    mc      = price_both(S, K, T, r, sigma, n_simulations=N_SIM, seed=42)

    rows.append({
        "Strike (K)": int(K),
        "Moneyness": "ITM" if K < S else ("ATM" if K == S else "OTM"),
        "BS Call (₹)": round(bs_call, 2),
        "MC Call (₹)": round(mc["call"].price, 2),
        "Call Δ (₹)": round(abs(bs_call - mc["call"].price), 2),
        "BS Put (₹)": round(bs_put, 2),
        "MC Put (₹)": round(mc["put"].price, 2),
        "Put Δ (₹)": round(abs(bs_put - mc["put"].price), 2),
        "MC Call SE": round(mc["call"].stderr, 3),
    })

df = pd.DataFrame(rows)
df.set_index("Strike (K)", inplace=True)
df

## 3. Call & Put Price vs Strike — Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col_bs, col_mc, title in [
    (axes[0], "BS Call (₹)", "MC Call (₹)", "European Call Price vs Strike"),
    (axes[1], "BS Put (₹)",  "MC Put (₹)",  "European Put Price vs Strike"),
]:
    ax.plot(df.index, df[col_bs], lw=2, label="Black-Scholes", color="#1f77b4")
    ax.plot(df.index, df[col_mc], lw=2, linestyle="--", label="Monte Carlo", color="#ff7f0e")
    ax.axvline(S, color="grey", linestyle=":", alpha=0.7, label=f"Spot = {S:,.0f}")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Strike (₹)")
    ax.set_ylabel("Option Price (₹)")
    ax.legend()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig("../results/bs_vs_mc_prices.png", dpi=150)
plt.show()

## 4. Greeks Surface — Call Options

In [ ]:
greek_names = ["delta", "gamma", "theta", "vega", "rho"]
K_range = np.linspace(18_000, 26_000, 200)

greek_values = {g: [] for g in greek_names}
for K in K_range:
    g = greeks(S, K, T, r, sigma, "call")
    for name in greek_names:
        greek_values[name].append(g[name])

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
colors = ["#1f77b4", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

for i, (name, color) in enumerate(zip(greek_names, colors)):
    axes[i].plot(K_range, greek_values[name], lw=2, color=color)
    axes[i].axvline(S, color="grey", linestyle=":", alpha=0.6)
    axes[i].set_title(name.capitalize(), fontsize=13)
    axes[i].set_xlabel("Strike (₹)")
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

axes[5].axis("off")
fig.suptitle("Black-Scholes Greeks vs Strike — European Call (Nifty 50 params)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../results/greeks_surface.png", dpi=150)
plt.show()

## 5. Implied Volatility Surface (Vol × Maturity Grid)

In [ ]:
vol_range = np.linspace(0.10, 0.50, 30)
T_range   = np.linspace(0.05, 1.0, 30)
K_fixed   = S  # ATM

VOL, MAT = np.meshgrid(vol_range, T_range)
PRICE = np.vectorize(lambda v, t: call_price(S, K_fixed, t, r, v))(VOL, MAT)

fig = plt.figure(figsize=(12, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(VOL * 100, MAT, PRICE, cmap=cm.viridis, alpha=0.9, linewidth=0)
fig.colorbar(surf, ax=ax, shrink=0.5, label="Call Price (₹)")
ax.set_xlabel("Implied Volatility (%)")
ax.set_ylabel("Time to Expiry (yr)")
ax.set_zlabel("Call Price (₹)")
ax.set_title("ATM Call Price Surface — Vol × Maturity", fontsize=13)
plt.tight_layout()
plt.savefig("../results/vol_surface.png", dpi=150)
plt.show()

## 6. Monte Carlo Convergence Analysis

In [ ]:
from src.options.monte_carlo import price_european_call

K_atm = S
bs_ref = call_price(S, K_atm, T, r, sigma)
sim_counts = [500, 1_000, 5_000, 10_000, 50_000, 100_000, 200_000]

mc_prices, mc_errors = [], []
for n in sim_counts:
    res = price_european_call(S, K_atm, T, r, sigma, n_simulations=n, seed=42)
    mc_prices.append(res.price)
    mc_errors.append(res.stderr * 1.96)

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(sim_counts, mc_prices, yerr=mc_errors, fmt="o-", capsize=4,
            color="#ff7f0e", label="MC Price ± 95% CI")
ax.axhline(bs_ref, color="#1f77b4", lw=2, linestyle="--", label=f"BS Reference = {bs_ref:.2f}")
ax.set_xscale("log")
ax.set_xlabel("Number of Simulations (log scale)")
ax.set_ylabel("ATM Call Price (₹)")
ax.set_title("Monte Carlo Convergence to Black-Scholes Price", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig("../results/mc_convergence.png", dpi=150)
plt.show()

conv_df = pd.DataFrame({
    "N Simulations": sim_counts,
    "MC Price (₹)": [round(p, 4) for p in mc_prices],
    "95% CI Width (₹)": [round(2 * e, 4) for e in mc_errors],
    "Error vs BS (₹)": [round(abs(p - bs_ref), 4) for p in mc_prices],
})
conv_df.set_index("N Simulations", inplace=True)
conv_df